In [382]:
import importlib
import os
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

import dtree_process_worker as _dtree_worker

# Force refresh in notebooks so newly added helpers are visible.
_dtree_worker = importlib.reload(_dtree_worker)

if hasattr(_dtree_worker, "build_class_folds_worker"):
    build_class_folds_worker = _dtree_worker.build_class_folds_worker
else:
    # Fallback keeps notebook runnable even if module is stale.
    def build_class_folds_worker(args):
        class_value, class_indices, k, seed = args
        indices = np.asarray(class_indices, dtype=np.int64).copy()
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
        split_indices = np.array_split(indices, int(k))
#         return class_value, [chunk.astype(np.int64) for chunk in split_indices]


In [383]:
def entropy_from_labels(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    if labels_arr.size == 0:
        return 0.0
    _, counts = np.unique(labels_arr, return_counts=True)
    probs = counts / counts.sum()
    return float(-np.sum(probs * np.log2(probs)))


def majority_label(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    values, counts = np.unique(labels_arr, return_counts=True)
    return int(values[np.argmax(counts)])


def split_dataset_np(dataset, feature_index, feature_value):
    data = np.asarray(dataset)
    if data.size == 0:
        return data
    mask = data[:, feature_index] == feature_value
    filtered = data[mask]
    if filtered.size == 0:
        return np.empty((0, data.shape[1] - 1), dtype=data.dtype)
    return np.delete(filtered, feature_index, axis=1)


def _feature_gain_worker(args):
    dataset, feature_index, base_entropy = args
    data = np.asarray(dataset)
    values = np.unique(data[:, feature_index])
    total = float(data.shape[0])
    conditional_entropy = 0.0

    for value in values:
        subset = split_dataset_np(data, feature_index, value)
        if subset.shape[0] == 0:
            continue
        conditional_entropy += (subset.shape[0] / total) * entropy_from_labels(subset[:, -1])

    return feature_index, base_entropy - conditional_entropy


def best_feature_threaded(dataset, thread_workers=8):
    data = np.asarray(dataset)
    n_features = data.shape[1] - 1
    if n_features <= 1:
        return 0

    base_entropy = entropy_from_labels(data[:, -1])
    args = np.empty(n_features, dtype=object)
    for idx in range(n_features):
        args[idx] = (data, idx, base_entropy)

    workers = max(1, min(int(thread_workers), n_features))
    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = tuple(executor.map(_feature_gain_worker, args))

    return int(max(results, key=lambda item: item[1])[0])


def build_tree(dataset, feature_names, min_sample_size=1, max_depth=None, depth=0, thread_workers=8):
    data = np.asarray(dataset)
    names = np.asarray(feature_names, dtype=object)

    labels = data[:, -1]

    # stop condition1: when number of samples less then setted
    if np.unique(labels).size <= max(1, min_sample_size):
        return majority_label(labels)

    # stop condition2: run out of features
    if data.shape[1] == 1:
        return majority_label(labels)

    # stop condition3: reach setted depth
    if max_depth is not None and depth >= max_depth:
        return majority_label(labels)

    

    best_feature = best_feature_threaded(data, thread_workers=thread_workers)
    root_name = names[best_feature]
    tree = {root_name: {}}

    child_feature_names = np.delete(names, best_feature)
    unique_values = np.unique(data[:, best_feature])
    for value in unique_values:
        child_data = split_dataset_np(data, best_feature, value)
        if child_data.shape[0] == 0:
            tree[root_name][int(value)] = majority_label(labels)
        else:
            tree[root_name][int(value)] = build_tree(
                child_data,
                child_feature_names,
                min_sample_size = min_sample_size,
                max_depth=max_depth,
                depth=depth + 1,
                thread_workers=thread_workers,
            )

    return tree


def predict_one(tree, feature_names, sample, default_label):
    node = tree
    names = np.asarray(feature_names, dtype=object)
    x = np.asarray(sample)

    while isinstance(node, dict):
        root = next(iter(node))
        children = node[root]

        matched = np.where(names == root)[0]
        if matched.size == 0:
            return default_label

        idx = int(matched[0])
        value = int(x[idx])
        if value not in children:
            return default_label

        node = children[value]
        names = np.delete(names, idx)
        x = np.delete(x, idx)

    return int(node)


def predict_batch(tree, feature_names, dataset, default_label):
    data = np.asarray(dataset)
    x = data[:, :-1]
    preds = np.empty(x.shape[0], dtype=np.int64)
    for i in range(x.shape[0]):
        preds[i] = predict_one(tree, feature_names, x[i], default_label)
    return preds


def accuracy_np(y_true, y_pred):
    true_arr = np.asarray(y_true, dtype=np.int64)
    pred_arr = np.asarray(y_pred, dtype=np.int64)
    if true_arr.size == 0:
        return 0.0
    return float(np.mean(true_arr == pred_arr))


def post_prune_tree_reduced_error(tree, train_data, valid_data, feature_names):
    if not isinstance(tree, dict):
        return tree

    train_np = np.asarray(train_data)
    valid_np = np.asarray(valid_data)
    names = np.asarray(feature_names, dtype=object)

    if train_np.shape[0] == 0:
        return tree

    root = next(iter(tree))
    children = tree[root]
    root_idx = int(np.where(names == root)[0][0])
    child_names = np.delete(names, root_idx)

    pruned_children = {}
    for edge_val, child in children.items():
        child_train = split_dataset_np(train_np, root_idx, edge_val)
        child_valid = split_dataset_np(valid_np, root_idx, edge_val)
        pruned_children[edge_val] = post_prune_tree_reduced_error(child, child_train, child_valid, child_names)

    pruned_tree = {root: pruned_children}

    if valid_np.shape[0] == 0:
        return pruned_tree

    default = majority_label(train_np[:, -1])
    subtree_pred = predict_batch(pruned_tree, names, valid_np, default)
    subtree_acc = accuracy_np(valid_np[:, -1], subtree_pred)

    leaf_pred = np.full(valid_np.shape[0], default, dtype=np.int64)
    leaf_acc = accuracy_np(valid_np[:, -1], leaf_pred)

    if leaf_acc >= subtree_acc:
        return int(default)

    return pruned_tree

In [384]:
def count_leaf_nodes(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    return int(sum(count_leaf_nodes(child) for child in tree[root].values()))


def tree_depth(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    child_depths = np.fromiter(
        (tree_depth(child) for child in tree[root].values()),
        dtype=np.int64,
    )
    return int(1 + child_depths.max(initial=0))


def save_markdown_report(results_array, output_path):
    rows = np.asarray(results_array, dtype=object)
    lines = np.empty(rows.shape[0] + 8, dtype=object)
    lines[0] = "# Fold Results"
    lines[1] = ""
    lines[2] = "| Fold | Unpruned Acc | Pruned Acc | Leaves | Depth |"
    lines[3] = "|---:|---:|---:|---:|---:|"

    unpruned = np.empty(rows.shape[0], dtype=np.float64)
    pruned = np.empty(rows.shape[0], dtype=np.float64)

    for i, item in enumerate(rows):
        unpruned[i] = float(item["unpruned_accuracy"])
        pruned[i] = float(item["pruned_accuracy"])
        lines[i + 4] = (
            f"| {item['fold']} | {unpruned[i]:.4f} | {pruned[i]:.4f} | "
            f"{item['leaf_count']} | {item['depth']} |"
        )

    lines[-4] = ""
    lines[-3] = f"- Mean unpruned accuracy: {unpruned.mean():.4f}"
    lines[-2] = f"- Mean pruned accuracy: {pruned.mean():.4f}"
    lines[-1] = ""

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text("\n".join(lines.tolist()), encoding="utf-8")
    print(f"Saved report: {output_file}")

# Datasets used in this notebook:
# 1) Letter Recognition
# 2) Adult
# 3) Mushroom
#
# Install once if needed:
# %pip install ucimlrepo

In [385]:
# UCI IDs: Letter=59, Adult=2, Mushroom=73
letter_df = fetch_ucirepo(id=59).data.original.copy()
adult_df = fetch_ucirepo(id=2).data.original.copy()
mushroom_df = fetch_ucirepo(id=73).data.original.copy()

print(letter_df.shape, adult_df.shape, mushroom_df.shape)
print("Letter columns:", letter_df.columns.to_numpy(dtype=object))


(20000, 17) (48842, 15) (8124, 23)
Letter columns: ['lettr' 'x-box' 'y-box' 'width' 'high' 'onpix' 'x-bar' 'y-bar' 'x2bar'
 'y2bar' 'xybar' 'x2ybr' 'xy2br' 'x-ege' 'xegvy' 'y-ege' 'yegvx']


In [386]:
def encode_dataframe_to_int_numpy(df, target_col):
    cols = df.columns.to_numpy(dtype=object)
    encoded = np.empty((df.shape[0], df.shape[1]), dtype=np.int64)

    for col_idx, col_name in enumerate(cols):
        series = df[col_name]
        if pd.api.types.is_numeric_dtype(series):
            encoded[:, col_idx] = pd.to_numeric(series, errors="coerce").fillna(0).to_numpy(dtype=np.int64)
        else:
            codes, _ = pd.factorize(series.astype(str), sort=True)
            encoded[:, col_idx] = codes.astype(np.int64)

    target_idx = int(np.where(cols == target_col)[0][0])
    feature_indices = np.delete(np.arange(cols.shape[0], dtype=np.int64), target_idx)
    feature_names = cols[feature_indices]

    dataset = np.concatenate(
        [encoded[:, feature_indices], encoded[:, target_idx:target_idx + 1]],
        axis=1,
    )
    return dataset, feature_names


def create_kfold_datasets_multiprocess(
    dataset,
    k=10,
    validation_ratio=0.1,
    sample_size_ratio=1.0,
    random_state=42,
    process_workers=None,
):
    data = np.asarray(dataset, dtype=np.int64)
    y = data[:, -1]

    rng = np.random.default_rng(random_state)
    class_values = np.unique(y)

    sampled_indices = np.empty(0, dtype=np.int64)
    for class_value in class_values:
        class_idx = np.where(y == class_value)[0]
        class_idx = rng.permutation(class_idx)
        if sample_size_ratio >= 1.0:
            take_count = class_idx.shape[0]
        else:
            take_count = max(1, int(class_idx.shape[0] * sample_size_ratio))
        sampled_indices = np.concatenate([sampled_indices, class_idx[:take_count]])

    sampled_indices = rng.permutation(sampled_indices)
    sampled_data = data[sampled_indices]

    validation_size = max(1, int(sampled_data.shape[0] * validation_ratio))
    validation_size = min(validation_size, sampled_data.shape[0] - 1)

    validation_data = sampled_data[:validation_size]
    kfold_data = sampled_data[validation_size:]
    y_kfold = kfold_data[:, -1]

    workers = process_workers
    if workers is None:
        workers = max(1, min(k, (os.cpu_count() or 1) - 1))

    task_args = np.empty(class_values.shape[0], dtype=object)
    for i, class_value in enumerate(class_values):
        class_local_idx = np.where(y_kfold == class_value)[0]
        task_args[i] = (int(class_value), class_local_idx, int(k), int(random_state + 1000 + i))

    with ProcessPoolExecutor(max_workers=workers) as executor:
        class_chunks = tuple(executor.map(build_class_folds_worker, task_args))

    fold_test_indices = np.empty(k, dtype=object)
    for i in range(k):
        fold_test_indices[i] = np.empty(0, dtype=np.int64)

    for _, chunks in class_chunks:
        for fold_idx in range(k):
            fold_test_indices[fold_idx] = np.concatenate([fold_test_indices[fold_idx], chunks[fold_idx]])

    for fold_idx in range(k):
        fold_test_indices[fold_idx] = rng.permutation(fold_test_indices[fold_idx])

    all_indices = np.arange(kfold_data.shape[0], dtype=np.int64)
    fold_records = np.empty(k, dtype=object)

    for fold_idx in range(k):
        test_idx = fold_test_indices[fold_idx]
        train_mask = np.ones(kfold_data.shape[0], dtype=bool)
        train_mask[test_idx] = False
        train_idx = all_indices[train_mask]

        fold_records[fold_idx] = {
            "fold": fold_idx + 1,
            "train": kfold_data[train_idx],
            "test": kfold_data[test_idx],
            "validation": validation_data,
        }

    return fold_records


letter_dataset, letter_dataset_feature_names = encode_dataframe_to_int_numpy(letter_df, target_col="lettr")
adult_dataset, adult_dataset_feature_names = encode_dataframe_to_int_numpy(adult_df, target_col="income")
mushroom_dataset, mushroom_dataset_feature_names = encode_dataframe_to_int_numpy(mushroom_df, target_col="poisonous")


letter_fold_datasets = create_kfold_datasets_multiprocess(
    letter_dataset,
    k=10,
    validation_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

adult_fold_datasets = create_kfold_datasets_multiprocess(
    adult_dataset,
    k=10,
    validation_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

mushroom_fold_datasets = create_kfold_datasets_multiprocess(
    mushroom_dataset,
    k=10,
    validation_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

original_datasets = [letter_dataset, adult_dataset, mushroom_dataset]
folded_datasets = [letter_fold_datasets, adult_fold_datasets, mushroom_fold_datasets]


for original_dataset, folded_dataset in zip(original_datasets, folded_datasets):
    print("dataset shape:", original_dataset.shape)
    print("validation size:", folded_dataset[0]["validation"].shape[0])
    for fold in folded_dataset:
        print(
            f"fold {fold['fold']}: train={fold['train'].shape[0]}, "
            f"test={fold['test'].shape[0]}, validation={fold['validation'].shape[0]}"
        )
    print("\n")


dataset shape: (20000, 17)
validation size: 2000
fold 1: train=16187, test=1813, validation=2000
fold 2: train=16195, test=1805, validation=2000
fold 3: train=16196, test=1804, validation=2000
fold 4: train=16197, test=1803, validation=2000
fold 5: train=16199, test=1801, validation=2000
fold 6: train=16202, test=1798, validation=2000
fold 7: train=16204, test=1796, validation=2000
fold 8: train=16204, test=1796, validation=2000
fold 9: train=16206, test=1794, validation=2000
fold 10: train=16210, test=1790, validation=2000


dataset shape: (48842, 15)
validation size: 4884
fold 1: train=39560, test=4398, validation=4884
fold 2: train=39561, test=4397, validation=4884
fold 3: train=39562, test=4396, validation=4884
fold 4: train=39562, test=4396, validation=4884
fold 5: train=39562, test=4396, validation=4884
fold 6: train=39563, test=4395, validation=4884
fold 7: train=39563, test=4395, validation=4884
fold 8: train=39563, test=4395, validation=4884
fold 9: train=39563, test=4395, val

# Train and evaluate all cases 

In [387]:
import sys
import time

def prepruning_grid_search_fold(
    fold_record,
    feature_names,
    min_sample_sizes,
    max_depths,
    thread_workers=8,
    show_process=True,
):
    t0 = time.perf_counter()
    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)
    validation_data = np.asarray(fold_record["validation"], dtype=np.int64)

    fold_id = fold_record.get("fold", "?")
    grid_total = len(list(min_sample_sizes)) * len(list(max_depths))
    if show_process:
        print(
            f"[fold {fold_id}] start | train={train_data.shape[0]} test={test_data.shape[0]} "
            f"val={validation_data.shape[0]} | grid={grid_total} configs",
            flush=True,
        )

    default_label = majority_label(train_data[:, -1])
    names = np.asarray(feature_names, dtype=object)

    best = {
        "min_sample_size": None,
        "max_depth": None,
        "test_accuracy_for_selection": -1.0,
        "tree": None,
    }

    for k, (mss, mxd) in enumerate(
        ((int(mss), mxd) for mss in min_sample_sizes for mxd in max_depths),
        start=1,
    ):
        tree = build_tree(
            train_data,
            names,
            min_sample_size=mss,
            max_depth=mxd,
            thread_workers=thread_workers,
        )
        test_pred = predict_batch(tree, feature_names, test_data, default_label)
        test_acc = accuracy_np(test_data[:, -1], test_pred)
        leaves = count_leaf_nodes(tree)
        if test_acc > best["test_accuracy_for_selection"] or (
            test_acc == best["test_accuracy_for_selection"]
            and best["tree"] is not None
            and leaves < count_leaf_nodes(best["tree"])
        ):
            best.update(
                min_sample_size=mss,
                max_depth=mxd,
                test_accuracy_for_selection=float(test_acc),
                tree=tree,
            )
        # Optional: sparse progress (e.g. every 10% of grid)
        if show_process and grid_total >= 50 and (k % max(1, grid_total // 10) == 0):
            print(f"[fold {fold_id}] grid {k}/{grid_total} ... best test(sel)={best['test_accuracy_for_selection']:.4f}", flush=True)

    tree = best["tree"]
    val_pred = predict_batch(tree, feature_names, validation_data, default_label)
    validation_acc = accuracy_np(validation_data[:, -1], val_pred)
    elapsed = time.perf_counter() - t0

    if show_process:
        print(
            f"[fold {fold_id}] done in {elapsed:.1f}s | "
            f"best mss={best['min_sample_size']} max_depth={best['max_depth']} | "
            f"test(sel)={best['test_accuracy_for_selection']:.4f} | "
            f"val(report)={float(validation_acc):.4f} | "
            f"leaves={count_leaf_nodes(tree)} depth={tree_depth(tree)}",
            flush=True,
        )

    return {
        "fold": fold_record,
        "best_min_sample_size": best["min_sample_size"],
        "best_max_depth": best["max_depth"],
        "test_accuracy_used_to_select_hyperparameters": best["test_accuracy_for_selection"],
        "validation_accuracy": float(validation_acc),
        "tree": tree,
        "leaf_count": count_leaf_nodes(tree),
        "depth": tree_depth(tree),
    }


def run_prepruning_all_folds(
    fold_datasets,
    feature_names,
    min_sample_sizes,
    max_depths,
    thread_workers=8,
):
    results = []
    for i in range(len(fold_datasets)):
        results.append(
            prepruning_grid_search_fold(
                fold_datasets[i],
                np.asarray(feature_names, dtype=object),
                min_sample_sizes=min_sample_sizes,
                max_depths=max_depths,
                thread_workers=thread_workers,
            )
        )
    val_accs = np.array([r["validation_accuracy"] for r in results], dtype=np.float64)
    test_sel = np.array(
        [r["test_accuracy_used_to_select_hyperparameters"] for r in results],
        dtype=np.float64,
    )
    return {
        "per_fold": results,
        "mean_validation_accuracy": float(val_accs.mean()),
        "std_validation_accuracy": float(val_accs.std(ddof=1)) if val_accs.size > 1 else 0.0,
        "mean_test_accuracy_for_selection": float(test_sel.mean()),
        "validation_accuracies": val_accs,
    }

### Stump vs unpruned vs post-pruned

- **Stump:** `max_depth=1`
- **Unpruned:** `min_sample_size=1`, `max_depth=None`
- **Pruned:** pre-tuning, grid search on min_sample_size and max_depth

In [390]:
# helpers for stump vs unpruned vs pre-pruned (no post-pruning)

import numpy as np
import pandas as pd

def _fit_and_score_case(
    train_data,
    eval_data,  # validation set
    feature_names,
    min_sample_size,
    max_depth,
    thread_workers=8,
):
    default_label = majority_label(train_data[:, -1])
    tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=int(min_sample_size),
        max_depth=max_depth,
        thread_workers=thread_workers,
    )
    eval_pred = predict_batch(tree, feature_names, eval_data, default_label)
    eval_acc = accuracy_np(eval_data[:, -1], eval_pred)
    return {
        "tree": tree,
        "validation_accuracy": float(eval_acc),
        "leaf_count": int(count_leaf_nodes(tree)),
        "depth": int(tree_depth(tree)),
    }


def evaluate_three_models_one_fold(
    fold_record,
    feature_names,
    min_sample_sizes,
    thread_workers=8,
):
    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)          # used only for pre-pruned selection
    val_data = np.asarray(fold_record["validation"], dtype=np.int64)     # final reported metric
    fold_id = int(fold_record["fold"])

    # 1) Stump: max_depth=1, report on validation
    stump = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=1,
        thread_workers=thread_workers,
    )

    # 2) Unpruned: min_sample_size=1, max_depth=None, report on validation
    unpruned = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=None,
        thread_workers=thread_workers,
    )

    # Dynamic max_depth search space based on unpruned tree depth
    unpruned_depth = int(unpruned["depth"])
    dynamic_max_depths = list(range(0, unpruned_depth + 1))
    print("max_depths:", dynamic_max_depths)

    # 3) Pre-pruned: grid search on TEST set, final report on VALIDATION set
    default_label = majority_label(train_data[:, -1])
    best = {
        "min_sample_size": None,
        "max_depth": None,
        "selection_test_accuracy": -1.0,
        "tree": None,
        "leaf_count": None,
        "depth": None,
    }

    # count total combinations
    num_grid_configs = len(list(min_sample_sizes)) * len(dynamic_max_depths)
    grid_checked = 0

    for mss in min_sample_sizes:
        for md in dynamic_max_depths:
            grid_checked += 1

            tree = build_tree(
                train_data,
                np.asarray(feature_names, dtype=object),
                min_sample_size=int(mss),
                max_depth=md,
                thread_workers=thread_workers,
            )

            # selection metric on test set
            test_pred = predict_batch(tree, feature_names, test_data, default_label)
            test_acc = float(accuracy_np(test_data[:, -1], test_pred))
            leaves = int(count_leaf_nodes(tree))
            depth = int(tree_depth(tree))
            depth_aligned = depth - 1

            # print most recently calculated parameters + result
            print(
                f"[fold {fold_id}] {grid_checked}/{num_grid_configs} "
                f"| recent mss={int(mss)} max_depth={md} "
                f"| test_acc={test_acc:.4f} leaves={leaves} depth={depth_aligned}",
                flush=True
            )

            # tie-break: higher selection metric, then fewer leaves, then lower depth
            is_better = (
                (test_acc > best["selection_test_accuracy"]) or
                (
                    test_acc == best["selection_test_accuracy"]
                    and best["leaf_count"] is not None
                    and leaves < best["leaf_count"]
                ) or
                (
                    test_acc == best["selection_test_accuracy"]
                    and leaves == best["leaf_count"]
                    and best["depth"] is not None
                    and depth < best["depth"]
                )
            )

            if is_better:
                best.update(
                    min_sample_size=int(mss),
                    max_depth=md,
                    selection_test_accuracy=test_acc,
                    tree=tree,
                    leaf_count=leaves,
                    depth=depth,
                )
                print(
                    f"  -> NEW BEST: mss={best['min_sample_size']} max_depth={best['max_depth']} "
                    f"| test(sel)={best['selection_test_accuracy']:.4f}",
                    flush=True
                )

    pre_tree = best["tree"]
    pre_val_pred = predict_batch(pre_tree, feature_names, val_data, default_label)
    pre_val_acc = float(accuracy_np(val_data[:, -1], pre_val_pred))

    return [
        {
            "fold": fold_id,
            "model": "stump",
            "validation_accuracy": stump["validation_accuracy"],
            "leaf_count": stump["leaf_count"],
            "depth": stump["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": 1,
            "selection_test_accuracy": np.nan,
        },
        {
            "fold": fold_id,
            "model": "unpruned",
            "validation_accuracy": unpruned["validation_accuracy"],
            "leaf_count": unpruned["leaf_count"],
            "depth": unpruned["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": None,
            "selection_test_accuracy": np.nan,
        },
        {
            "fold": fold_id,
            "model": "pre_pruned",
            "validation_accuracy": pre_val_acc,
            "leaf_count": best["leaf_count"],
            "depth": best["depth"],
            "best_min_sample_size": best["min_sample_size"],
            "best_max_depth": best["max_depth"],
            "selection_test_accuracy": best["selection_test_accuracy"],
        },
    ]

In [ ]:
# run all datasets and summarize

from IPython.display import display

# You can widen/narrow this grid depending on runtime.
# Keep None if you want "no depth cap" to be a candidate in pre-pruning.
min_sample_sizes = list(range(1, 10, 1))

configs = [
    ("letter", letter_fold_datasets, letter_dataset_feature_names),
    ("adult", adult_fold_datasets, adult_dataset_feature_names),
    ("mushroom", mushroom_fold_datasets, mushroom_dataset_feature_names),
]

all_rows = []

for dataset_name, folds, feature_names in configs:
    print(f"\n=== {dataset_name} ===")
    for fold_record in folds:
        fold_rows = evaluate_three_models_one_fold(
            fold_record=fold_record,
            feature_names=feature_names,
            min_sample_sizes=min_sample_sizes,
            thread_workers=8,
        )
        for r in fold_rows:
            r["dataset"] = dataset_name
            all_rows.append(r)

results_df = pd.DataFrame(all_rows)
results_df["depth_aligned_to_max_depth"] = results_df["depth"] - 1


# Per-fold detailed results
display(results_df.sort_values(["dataset", "fold", "model"]).reset_index(drop=True))

# Mean/std comparison table
summary_df = (
    results_df
    .groupby(["dataset", "model"], as_index=False)
    .agg(
        mean_test_accuracy=("test_accuracy", "mean"),
        std_test_accuracy=("test_accuracy", "std"),
        mean_leaf_count=("leaf_count", "mean"),
        mean_depth=("depth_aligned_to_max_depth", "mean"),
    )
    .sort_values(["dataset", "mean_test_accuracy"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n=== Summary (mean ± std test accuracy) ===")
display(summary_df)

# Optional: show pre-pruned chosen hyperparameters frequency
pre_choices = (
    results_df[results_df["model"] == "pre_pruned"]
    .groupby(["dataset", "best_min_sample_size", "best_max_depth"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["dataset", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n=== Pre-pruned hyperparameter choices ===")
display(pre_choices)


=== letter ===
max_depths: [0, 1, 2, 3, 4, 5, 6, 7, 8]
[fold 1] 1/81 | recent mss=1 max_depth=0 | test_acc=0.0408 leaves=1 depth=0
  -> NEW BEST: mss=1 max_depth=0 | test(sel)=0.0408
[fold 1] 2/81 | recent mss=1 max_depth=1 | test_acc=0.1804 leaves=16 depth=1
  -> NEW BEST: mss=1 max_depth=1 | test(sel)=0.1804
[fold 1] 3/81 | recent mss=1 max_depth=2 | test_acc=0.4109 leaves=182 depth=2
  -> NEW BEST: mss=1 max_depth=2 | test(sel)=0.4109
[fold 1] 4/81 | recent mss=1 max_depth=3 | test_acc=0.6106 leaves=1166 depth=3
  -> NEW BEST: mss=1 max_depth=3 | test(sel)=0.6106
[fold 1] 5/81 | recent mss=1 max_depth=4 | test_acc=0.7341 leaves=3642 depth=4
  -> NEW BEST: mss=1 max_depth=4 | test(sel)=0.7341
[fold 1] 6/81 | recent mss=1 max_depth=5 | test_acc=0.7341 leaves=5633 depth=5
[fold 1] 7/81 | recent mss=1 max_depth=6 | test_acc=0.7341 leaves=5947 depth=6
[fold 1] 8/81 | recent mss=1 max_depth=7 | test_acc=0.7336 leaves=5955 depth=7
[fold 1] 9/81 | recent mss=1 max_depth=8 | test_acc=0.7336